# I-E03 — I-N120 + I-S50 / I-S101 Soft-Voting

I-S50은 DiceCE + Deep Supervision 모델이며, SegResNet 중 lesion recall이 가장 높아 미탐 보완 후보로 평가한다.

후보:

- `N120 0.7 + S50 0.3`
- `N120 0.6 + S101 0.2 + S50 0.2`
- `N120 0.5 + S101 0.25 + S50 0.25`

동일 38-case validation에서 probability map을 물리 좌표계로 정렬한 뒤 soft-voting하고, `FP/scan ≤ 1.5` 안에서 lesion F2가 최대인 threshold를 고른다.


In [ ]:
from pathlib import Path
import json, subprocess
import pandas as pd
from IPython.display import display

REPO = Path("/home/stu03/workspace/model")
PYTHON = REPO / ".venv/bin/python"
LABEL_DIR = Path("/home/stu03/workspace/datasets/ischemic/nnunet/nnUNet_preprocessed/Dataset502_ISLES2022/gt_segmentations")
N120_PROB = REPO / "runs/I-nnunet/Dataset502_ISLES2022/nnUNetTrainerIschemiaMultimodal__IschemiaPlansDualEncoderGatedV1__3d_fullres/fold_0/best_probability_validation"
S101_PROB = REPO / "runs/I-segresnet/I-S101/final_metrics/val_probabilities"
S101_REF = REPO / "runs/I-segresnet/I-S101/final_metrics/val_predictions"
S50_PROB = REPO / "runs/I-segresnet/I-S50/final_metrics/val_probabilities"
S50_REF = REPO / "runs/I-segresnet/I-S50/final_metrics/val_predictions"
OUT_ROOT = REPO / "runs/ischemia_ensemble/I-E03_N120_S50_S101"
CANDIDATES = [
    ("N120_070_S50_030", [("I-N120", N120_PROB, None, .70), ("I-S50", S50_PROB, S50_REF, .30)]),
    ("N120_060_S101_020_S50_020", [("I-N120", N120_PROB, None, .60), ("I-S101", S101_PROB, S101_REF, .20), ("I-S50", S50_PROB, S50_REF, .20)]),
    ("N120_050_S101_025_S50_025", [("I-N120", N120_PROB, None, .50), ("I-S101", S101_PROB, S101_REF, .25), ("I-S50", S50_PROB, S50_REF, .25)]),
]
assert all(x.exists() for x in [PYTHON, LABEL_DIR, N120_PROB, S101_PROB, S101_REF, S50_PROB, S50_REF])
[(name, [(model, weight) for model, _, _, weight in models]) for name, models in CANDIDATES]


## Soft-voting 실행


In [ ]:
for tag, models in CANDIDATES:
    command = [str(PYTHON), str(REPO / "scripts/ischemia/ensemble/evaluate_soft_voting.py")]
    for name, probability_dir, reference_dir, weight in models:
        spec = str(probability_dir) if reference_dir is None else f"{probability_dir}::{reference_dir}"
        command += ["--model", f"{name}={spec}", "--weight", str(weight)]
    command += ["--label-dir", str(LABEL_DIR), "--output-dir", str(OUT_ROOT / tag), "--max-fp-per-scan", "1.5"]
    print(f"[RUN] {tag}")
    subprocess.run(command, cwd=REPO, check=True)


## 후보 비교


In [ ]:
rows = []
for tag, _ in CANDIDATES:
    summary = json.loads((OUT_ROOT / tag / "ensemble_summary.json").read_text())
    rows.append({"candidate": tag, **{f"weight_{name}": weight for name, weight in summary["weights"].items()}, **summary["selected"]})
comparison = pd.DataFrame(rows).sort_values(["lesion_f2", "macro_dice_positive"], ascending=False)
display(comparison[[
    "candidate", "threshold", "macro_dice_positive", "voxel_recall", "voxel_precision",
    "lesion_recall", "lesion_precision", "lesion_f1", "lesion_f2",
    "small_lesion_recall", "fp_per_scan", "negative_case_fp_rate",
]])
print("추천: lesion F2 우선, 동률이면 Dice가 높은 첫 번째 행")


## 최종 후보 threshold 확인


In [ ]:
best = comparison.iloc[0]
sweep = pd.read_csv(OUT_ROOT / best["candidate"] / "ensemble_threshold_sweep.csv")
ensemble = sweep[sweep["candidate"].str.startswith("soft_vote__")]
display(ensemble[[
    "threshold", "macro_dice_positive", "voxel_recall", "voxel_precision",
    "lesion_recall", "lesion_precision", "lesion_f2", "small_lesion_recall", "fp_per_scan",
]].sort_values("threshold"))
